# Step 1: Install PySpark

In [1]:
!pip install pyspark


# Step 2: Upload CSV files

In [2]:
from google.colab import files
uploaded = files.upload()


Saving inventory_table.csv to inventory_table.csv
Saving orders_table.csv to orders_table.csv
Saving suppliers_table.csv to suppliers_table.csv


# Step 3: Data Cleaning & Processing

In [8]:
import pandas as pd
import numpy as np

# Load CSVs
orders_df = pd.read_csv("orders_table.csv")
suppliers_df = pd.read_csv("suppliers_table.csv")
inventory_df = pd.read_csv("inventory_table.csv")

print("=== Raw Orders Data ===")
print(orders_df.head(), "\n")

# Clean column names
orders_df.columns = orders_df.columns.str.strip().str.lower()
suppliers_df.columns = suppliers_df.columns.str.strip().str.lower()
inventory_df.columns = inventory_df.columns.str.strip().str.lower()

# Drop nulls
orders_df.dropna(inplace=True)
suppliers_df.dropna(inplace=True)
inventory_df.dropna(inplace=True)

print("===Cleaned Orders Data ===")
print(orders_df.head(), "\n")

# Convert dates
orders_df['order_date'] = pd.to_datetime(orders_df['order_date'], errors="coerce")
orders_df['delivery_date'] = pd.to_datetime(orders_df['delivery_date'], errors="coerce")

# Delay calculations
orders_df['delay_days'] = (orders_df['delivery_date'] - orders_df['order_date']).dt.days
orders_df['is_delayed'] = np.where(orders_df['delay_days'] > 1, 1, 0)

print("===Delay Calculations ===")
print(orders_df[['order_id','supplier_id','delay_days','is_delayed']].head(), "\n")



=== Raw Orders Data ===
   order_id  supplier_id  inventory_id  order_date delivery_date     status  \
0         1            1             1  2025-07-01    2025-07-03  Delivered   
1         2            2             2  2025-07-05    2025-07-23  Delivered   
2         3            2             3  2025-07-06    2025-07-09  Delivered   
3         4            3             4  2025-07-10           NaN    Shipped   
4         5            1             5  2025-07-12           NaN    Pending   

   quantity  
0        30  
1        10  
2        20  
3         5  
4        12   

===Cleaned Orders Data ===
   order_id  supplier_id  inventory_id  order_date delivery_date     status  \
0         1            1             1  2025-07-01    2025-07-03  Delivered   
1         2            2             2  2025-07-05    2025-07-23  Delivered   
2         3            2             3  2025-07-06    2025-07-09  Delivered   

   quantity  
0        30  
1        10  
2        20   

===Delay Calc

# Step 4: PySpark – Filter Delayed Shipments

In [9]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, datediff, current_date

spark = SparkSession.builder.appName("Week3").getOrCreate()

# Load orders with Spark
orders_spark = spark.read.csv("orders_table.csv", header=True, inferSchema=True)

# Convert delivery_date to DateType
orders_spark = orders_spark.withColumn("delivery_date", col("delivery_date").cast("date"))

# Calculate delay
orders_spark = orders_spark.withColumn("delay_days", datediff(current_date(), col("delivery_date")))

# Filter delayed shipments
delayed_df = orders_spark.filter(col("delay_days") > 0)

print("Delayed Shipments:")
delayed_df.show(5)


Delayed Shipments:
+--------+-----------+------------+----------+-------------+---------+--------+----------+
|order_id|supplier_id|inventory_id|order_date|delivery_date|   status|quantity|delay_days|
+--------+-----------+------------+----------+-------------+---------+--------+----------+
|       1|          1|           1|2025-07-01|   2025-07-03|Delivered|      30|        55|
|       2|          2|           2|2025-07-05|   2025-07-23|Delivered|      10|        35|
|       3|          2|           3|2025-07-06|   2025-07-09|Delivered|      20|        49|
+--------+-----------+------------+----------+-------------+---------+--------+----------+



# Step 5: Group by Supplier

In [10]:
grouped_df = delayed_df.groupBy("supplier_id").count().withColumnRenamed("count", "delayed_orders_count")

print("Delayed Orders by Supplier:")
grouped_df.show()


Delayed Orders by Supplier:
+-----------+--------------------+
|supplier_id|delayed_orders_count|
+-----------+--------------------+
|          1|                   1|
|          2|                   2|
+-----------+--------------------+



# Step 6: Data Cleaning in PySpark

In [12]:
from pyspark.sql.functions import to_date, trim, upper

orders_clean = (orders_spark
    .withColumn("order_date", to_date(col("order_date")))
    .withColumn("delivery_date", to_date(col("delivery_date")))
    .withColumn("status", upper(trim(col("status"))))
)

# Filter only delivered orders
delivered_df = orders_clean.filter(col("status") == "DELIVERED")

print("Week 4 Results – Delivered Orders:")
delivered_df.show(5)


Week 4 Results – Delivered Orders:
+--------+-----------+------------+----------+-------------+---------+--------+----------+
|order_id|supplier_id|inventory_id|order_date|delivery_date|   status|quantity|delay_days|
+--------+-----------+------------+----------+-------------+---------+--------+----------+
|       1|          1|           1|2025-07-01|   2025-07-03|DELIVERED|      30|        55|
|       2|          2|           2|2025-07-05|   2025-07-23|DELIVERED|      10|        35|
|       3|          2|           3|2025-07-06|   2025-07-09|DELIVERED|      20|        49|
+--------+-----------+------------+----------+-------------+---------+--------+----------+



# Step 7: Analysis Queries in SQL

In [13]:
# Register temp view
orders_clean.createOrReplaceTempView("orders")

# Example analyses
print("Top 3 Suppliers by Avg Order Quantity:")
spark.sql("""
    SELECT supplier_id, AVG(quantity) AS avg_quantity
    FROM orders
    GROUP BY supplier_id
    ORDER BY avg_quantity DESC
    LIMIT 3
""").show()

print("Orders Delayed > 5 Days:")
spark.sql("""
    SELECT order_id, supplier_id, DATEDIFF(delivery_date, order_date) AS delay_days
    FROM orders
    WHERE DATEDIFF(delivery_date, order_date) > 5
    ORDER BY delay_days DESC
""").show()


Top 3 Suppliers by Avg Order Quantity:
+-----------+------------+
|supplier_id|avg_quantity|
+-----------+------------+
|          1|        20.5|
|          2|        17.5|
|          3|         5.0|
+-----------+------------+

Orders Delayed > 5 Days:
+--------+-----------+----------+
|order_id|supplier_id|delay_days|
+--------+-----------+----------+
|       2|          2|        18|
+--------+-----------+----------+



Step 8: Save Cleaned Output

In [15]:
# Save as single CSV file in one partition
orders_clean.coalesce(1).write.option("header", "true").mode("overwrite").csv("/content/orders_cleaned_csv")

# Find the actual part file
import glob, shutil

part_file = glob.glob("/content/orders_cleaned_csv/part-*.csv")[0]
final_path = "/content/orders_cleaned.csv"

# Rename/move to a cleaner file name
shutil.move(part_file, final_path)

print(f"Cleaned CSV saved at: {final_path}")

# Download in Colab
from google.colab import files
files.download(final_path)


Cleaned CSV saved at: /content/orders_cleaned.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>